# Ablation 2 — Retrieval Strategy (CLPsych 2026 Task 1)

Evaluates the impact of in-context example retrieval. Prompting (P1+P2+P3) and ensemble (S3-merge) are held constant; only the retrieval strategy varies.

**Configurations**
- `ablation2_zeroshot` — no in-context examples
- `ablation2_cosine_top8` — top-8 by cosine similarity (no diversity re-ranking)

The third row of the ablation table (MMR top-8, α=0.6) reuses the existing **S3** submission, which uses identical parameters.

**Total: 2 new Codabench submissions.**

## Run order
1. Run cells 1–6 (Qwen, both configs).
2. Restart runtime.
3. Re-run cells 1–4, then run the Mistral cell (both configs).
4. Run the build cell to produce the 2 submission zips.

## Notes
- In `RAGIndex.retrieve(...)`, `diversity_alpha` weights `α * similarity + (1-α) * coverage_gain`. So **α = 1.0 → pure cosine top-k** (no diversity), α = 0.6 → MMR (our main system), α = 0.0 → pure diversity. We use `RAG_ALPHA_COSINE = 1.0` here.
- `build_messages(...)`, `build_messages_adaptive(...)`, and `build_messages_per_element(...)` accept an empty examples list `[]` for the zero-shot config.
- `retrieve_for_element(...)` has no MMR knob, so cosine and zero-shot share P3 retrieval logic; only zero-shot zeros it out.

In [ ]:
# 1. Setup
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # repo root: local modules + requirements.txt live here
sys.path.insert(0, os.getcwd())

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader  # optional: confirms a GPU is available
%pip install -q -r requirements.txt
%pip uninstall -y -q torchcodec

In [ ]:
MODEL_1 = 'Qwen/Qwen3.5-27B'
MODEL_2 = 'mistralai/Mistral-Small-3.2-24B-Instruct-2506'
TEMPERATURE = 0.1
MAX_TOKENS = 100
RAG_K = 8
RAG_POOL = 20            # match S3 main system
RAG_ALPHA_COSINE = 1.0   # α=1.0 → pure cosine top-k (no diversity)

RETRIEVAL_CONFIGS = ['zeroshot', 'cosine_top8']

In [ ]:
  #patch

  import os
  os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
  import rag_index, gc, torch
  from sentence_transformers import SentenceTransformer

  rag_index._model_cache.clear()
  gc.collect();

  def _get_model_safe(model_name="Alibaba-NLP/gte-multilingual-base"):
      if model_name not in rag_index._model_cache:
          m = SentenceTransformer(model_name, trust_remote_code=True, device='cpu')
          m.max_seq_length = 512
          # Fix: position_ids buffer не инициализирован в этой версии transformers.
          emb = m[0].auto_model.embeddings
          if hasattr(emb, 'position_ids') and emb.position_ids is not None:
              n = emb.position_ids.shape[-1]
              emb.position_ids = torch.arange(n, dtype=torch.long).unsqueeze(0)
              print(f'Fixed position_ids: shape={emb.position_ids.shape}, range=[0..{n-1}]')
          else:
              # Buffer не существует — создаём
              emb.register_buffer('position_ids', torch.arange(8192, dtype=torch.long).unsqueeze(0), persistent=False)
              print('Created position_ids buffer.')
          rag_index._model_cache[model_name] = m
      return rag_index._model_cache[model_name]

  def _embed_texts_safe(texts, model_name="Alibaba-NLP/gte-multilingual-base"):
      model = _get_model_safe(model_name)
      return model.encode(texts, normalize_embeddings=True, show_progress_bar=True,
                          batch_size=8, convert_to_numpy=True)

  rag_index._get_model = _get_model_safe
  rag_index._embed_texts = _embed_texts_safe
  print('Patched.')

In [ ]:
from data_loader import load_all_timelines
from rag_index import RAGIndex
from prompt_builder_rag import build_messages, build_messages_adaptive, build_messages_per_element
from output_parser import parse_llm_output
from tqdm import tqdm
import json, gc, torch
from pathlib import Path
from vllm import LLM, SamplingParams

# Build RAG index from train
train_posts = load_all_timelines('train_tasks12')
rag_index_real = RAGIndex([p for p in train_posts if p.has_evidence])

# Load test
test_posts = load_all_timelines('test_tasks12nolabels')
print(f'Test posts: {len(test_posts)}')

def retrieve_unified(text, timeline_id, mode):
    if mode == 'zeroshot':
        return []
    # cosine_top8 is the only retrieval mode here; MMR rerun is reused from S3.
    alpha = RAG_ALPHA_COSINE
    return rag_index_real.retrieve(text, k=RAG_K, exclude_timeline=timeline_id,
                                   candidate_pool=RAG_POOL, diversity_alpha=alpha)

def retrieve_element_A(text, timeline_id, mode):
    if mode == 'zeroshot':
        return []
    return rag_index_real.retrieve_for_element(text, 'A', k=RAG_K, exclude_timeline=timeline_id)

# Per-config: P1 messages, P3 (per-element A) messages, and the RAG cache used to build P2 later.
prompts = {}
for mode in RETRIEVAL_CONFIGS:
    p1_msgs, rag_cache, p3_msgs = [], [], []
    for p in tqdm(test_posts, desc=f'{mode} prompts'):
        rag = retrieve_unified(p.text, p.timeline_id, mode)
        rag_cache.append(rag)
        p1_msgs.append(build_messages(p.text, rag))
        rag_A = retrieve_element_A(p.text, p.timeline_id, mode)
        p3_msgs.append(build_messages_per_element(p.text, 'A', rag_A))
    prompts[mode] = {'p1_msgs': p1_msgs, 'p3_msgs': p3_msgs, 'rag_cache': rag_cache}
print('Built prompts for all retrieval configs.')

In [ ]:
def run_pipeline_for_config(llm, params, chat_kwargs, p1_msgs, p3_msgs, rag_cache):
    """P1 -> P2 (built from this config's RAG + P1 maladaptive) -> P3."""
    out_p1 = llm.chat(p1_msgs, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p1 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p1]

    msgs_p2 = [build_messages_adaptive(p.text, rag_cache[i], parsed_p1[i]['maladaptive'])
               for i, p in enumerate(test_posts)]
    out_p2 = llm.chat(msgs_p2, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p2 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p2]

    out_p3 = llm.chat(p3_msgs, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p3 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p3]
    return parsed_p1, parsed_p2, parsed_p3

def run_all_configs(model_name):
    print(f'\nLoading {model_name}...')
    llm = LLM(model_name, dtype='float16', max_model_len=16384, tensor_parallel_size=1, enforce_eager=True)
    params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
    chat_kwargs = {'enable_thinking': False} if 'qwen' in model_name.lower() else {}
    results = {}
    for mode in RETRIEVAL_CONFIGS:
        print(f'\n--- {mode} ---')
        p1, p2, p3 = run_pipeline_for_config(
            llm, params, chat_kwargs,
            prompts[mode]['p1_msgs'], prompts[mode]['p3_msgs'], prompts[mode]['rag_cache'])
        results[mode] = {'p1': p1, 'p2': p2, 'p3': p3}
    del llm; gc.collect(); torch.cuda.empty_cache()
    return results

qwen_results = run_all_configs(MODEL_1)

Path('outputs').mkdir(exist_ok=True)
with open('outputs/ablation2_qwen_raw.json', 'w') as f:
    json.dump(qwen_results, f)
print('Saved outputs/ablation2_qwen_raw.json')
print('\n>>> Restart runtime, re-run cells 1–4, then run the next cell. <<<')

In [ ]:
!nvidia-smi

---
## After runtime restart
Re-run cells 1, 2, 3, 4 (skip cell 5 — vLLM not loaded yet), then run the next cell.
Optional: free disk by removing the Qwen weights cache before pulling Mistral:
```
!rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3.5-27B
```

In [ ]:
!rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3.5-27B

In [ ]:
  #patch

  import os
  os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
  print('vLLM будет использовать spawn (избегает CUDA fork bug).')

In [ ]:
# Run Mistral (after restart). Loads Qwen raw outputs from disk.
import rag_index
rag_index._model_cache.clear()
gc.collect(); torch.cuda.empty_cache()

with open('outputs/ablation2_qwen_raw.json') as f:
    qwen_results = json.load(f)
print(f'Loaded Qwen raw for configs: {list(qwen_results.keys())}')

# Re-define helpers in case the kernel was restarted.
def run_pipeline_for_config(llm, params, chat_kwargs, p1_msgs, p3_msgs, rag_cache):
    out_p1 = llm.chat(p1_msgs, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p1 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p1]
    msgs_p2 = [build_messages_adaptive(p.text, rag_cache[i], parsed_p1[i]['maladaptive'])
               for i, p in enumerate(test_posts)]
    out_p2 = llm.chat(msgs_p2, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p2 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p2]
    out_p3 = llm.chat(p3_msgs, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p3 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p3]
    return parsed_p1, parsed_p2, parsed_p3

def run_all_configs(model_name):
    print(f'\nLoading {model_name}...')
    llm = LLM(model_name, dtype='float16', max_model_len=16384, tensor_parallel_size=1, enforce_eager=True)
    params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
    chat_kwargs = {'enable_thinking': False} if 'qwen' in model_name.lower() else {}
    results = {}
    for mode in RETRIEVAL_CONFIGS:
        print(f'\n--- {mode} ---')
        p1, p2, p3 = run_pipeline_for_config(
            llm, params, chat_kwargs,
            prompts[mode]['p1_msgs'], prompts[mode]['p3_msgs'], prompts[mode]['rag_cache'])
        results[mode] = {'p1': p1, 'p2': p2, 'p3': p3}
    del llm; gc.collect(); torch.cuda.empty_cache()
    return results

mistral_results = run_all_configs(MODEL_2)
with open('outputs/ablation2_mistral_raw.json', 'w') as f:
    json.dump(mistral_results, f)
print('Saved outputs/ablation2_mistral_raw.json')

In [ ]:
# Build the 2 ablation submissions from cached raw outputs.
import json, zipfile, sys, subprocess
from pathlib import Path
from submission_formatter import save_submission

with open('outputs/ablation2_qwen_raw.json') as f:
    qwen_results = json.load(f)
with open('outputs/ablation2_mistral_raw.json') as f:
    mistral_results = json.load(f)

def build_within_model(unified, adaptive=None, per_a=None):
    """Combine raw call outputs into a single-model prediction.
    Maladaptive + adaptive-presence from unified; adaptive elements from P2 if available;
    Affect overridden by P3 if available. (Same logic as the main pipeline.)"""
    merged = {
        'maladaptive': {'elements': dict(unified['maladaptive']['elements']),
                        'presence': unified['maladaptive']['presence']},
        'adaptive': {'elements': dict(adaptive['adaptive']['elements']) if adaptive else dict(unified['adaptive']['elements']),
                     'presence': unified['adaptive']['presence']},
    }
    if per_a is not None:
        for v in ('adaptive', 'maladaptive'):
            merged[v]['elements'].pop('A', None)
            if 'A' in per_a[v]['elements']:
                merged[v]['elements']['A'] = per_a[v]['elements']['A']
    for v in ('adaptive', 'maladaptive'):
        if not merged[v]['elements']:
            merged[v]['presence'] = 1
    return merged

def merge_ensemble_s3(qpred, mpred):
    """S3-merge: Qwen elements only, averaged presence (with the {1,2}=>2 rule)."""
    merged = {'maladaptive': {'elements': {}, 'presence': 1},
              'adaptive': {'elements': {}, 'presence': 1}}
    for v in ('maladaptive', 'adaptive'):
        pq, pg = qpred[v]['presence'], mpred[v]['presence']
        if pq == pg: merged[v]['presence'] = pq
        elif {pq, pg} == {1, 2}: merged[v]['presence'] = 2
        else: merged[v]['presence'] = round((pq + pg) / 2)
        merged[v]['elements'] = dict(qpred[v]['elements'])
        if not merged[v]['elements']: merged[v]['presence'] = 1
    return merged

# Both configs use the same backbone: full P1+P2+P3 within-model + S3-merge ensemble.
def make_predictor(mode):
    q, m = qwen_results[mode], mistral_results[mode]
    return lambda i: merge_ensemble_s3(
        build_within_model(q['p1'][i], q['p2'][i], q['p3'][i]),
        build_within_model(m['p1'][i], m['p2'][i], m['p3'][i]))

configs = {
    f'ablation2_{mode}': (f'{mode} · P1+P2+P3 · ensemble (S3-merge)', make_predictor(mode))
    for mode in RETRIEVAL_CONFIGS
}

Path('outputs').mkdir(exist_ok=True)
if 'official_eval' not in sys.path:
    sys.path.insert(0, 'official_eval')
for name, (description, fn) in configs.items():
    print(f'\n=== {description} ===')
    final = []
    for i, p in enumerate(test_posts):
        final.append({'timeline_id': p.timeline_id, 'post_id': p.post_id, 'prediction': fn(i)})

    raw_path = f'outputs/{name}_raw.json'
    pred_path = f'outputs/{name}_task1_pred.json'
    with open(raw_path, 'w') as f:
        json.dump(final, f, indent=2)
    save_submission(final, pred_path)

    result = subprocess.run(
        [sys.executable, 'official_eval/validate_submission.py',
         '--task1', pred_path, '--test-dir', 'test_tasks12nolabels'],
        capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode != 0:
        print('STDERR:', result.stderr)

    zip_path = f'outputs/{name}.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        zf.write(pred_path, 'task1_pred.json')

print('\nDone! 2 submission zips saved to outputs/.')